# Task 4: Advanced Model — BERT Sentiment Classifier
**Smart Review Analyzer**

- Model: `distilbert-base-uncased` (أخف وأسرع من BERT الكامل وقريب في الدقة)
- Dataset: Amazon Reviews
- Input: `text_splits.csv` من outputs الـ Feature Extraction


## 0. Install & Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)

from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_class_weight

# ── Device ───────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# ── Reproducibility ──────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print('All imports successful ✓')
print("done")

In [ ]:
import re
import string
import pandas as pd

# Optional NLP tools
import nltk
from nltk.corpus import stopwords

# Ensure stopwords are available
try:
    STOPWORDS = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    STOPWORDS = set(stopwords.words("english"))


# -----------------------------
# Text Cleaning Functions
# -----------------------------

def clean_text(text: str) -> str:
    """
    Basic text cleaning:
    - Lowercasing
    - Removing punctuation
    - Removing numbers
    - Removing extra spaces
    """
    if not isinstance(text, str):
        return ""

    text = text.lower()

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


def remove_stopwords(text: str) -> str:
    """
    Remove English stopwords
    """
    tokens = text.split()
    filtered_tokens = [word for word in tokens if word not in STOPWORDS]
    return " ".join(filtered_tokens)


def preprocess_text(text: str) -> str:
    """
    Full preprocessing pipeline for a single text
    """
    text = clean_text(text)
    text = remove_stopwords(text)
    return text


# -----------------------------
# Dataset-Level Processing
# -----------------------------

def preprocess_dataset(data: pd.DataFrame) -> pd.DataFrame:
    """
    Apply full preprocessing pipeline to dataset.

    Expected columns:
    - 'Review Title'
    - 'Review Text'
    - 'Rating'

    Output:
    - Adds 'Full Review'
    - Adds 'Sentiment'
    - Adds 'Cleaned Review'
    """

    # Drop unnecessary columns safely
    columns_to_drop = [
        'Reviewer Name',
        'Profile Link',
        'Review Count',
        'Review Date',
        'Date of Experience'
    ]
    data = data.drop(columns=[col for col in columns_to_drop if col in data.columns], errors='ignore')

    # Handle missing values
    data = data.dropna(subset=['Review Text', 'Rating'])
    data['Review Title'] = data['Review Title'].fillna('')
    data['Country'] = data['Country'].fillna('Unknown')

    # Extract numeric rating
    data['Rating'] = data['Rating'].astype(str).str.extract(r'(\d)').astype(float)
    data = data.dropna(subset=['Rating'])
    data['Rating'] = data['Rating'].astype(int)

    # Convert rating to sentiment
    def rating_to_sentiment(rating):
        if rating <= 2:
            return 0  # Negative
        elif rating >= 4:
            return 1  # Positive
        else:
            return None  # Neutral (drop later)

    data['Sentiment'] = data['Rating'].apply(rating_to_sentiment)
    data = data.dropna(subset=['Sentiment'])
    data['Sentiment'] = data['Sentiment'].astype(int)

    # Combine title + text
    data['Full Review'] = data['Review Title'] + " " + data['Review Text']

    # Apply text preprocessing
    data['Cleaned Review'] = data['Full Review'].apply(preprocess_text)

    # Optional: Review length (useful for analysis)
    data['Review Length'] = data['Cleaned Review'].apply(lambda x: len(x.split()))

    return data


# -----------------------------
# Utility Function
# -----------------------------

def save_processed_data(data: pd.DataFrame, output_path: str):
    """
    Save processed dataset to CSV
    """
    data.to_csv(output_path, index=False)
    print(f"Processed data saved to: {output_path}")


# -----------------------------
# Quick Test (Optional)
# -----------------------------
if __name__ == "__main__":
    # Example usage
    input_path = "/kaggle/input/datasets/dongrelaxman/amazon-reviews-dataset/Amazon_Reviews.csv"
    output_path = "/kaggle/working/clean_reviews.csv"

    df = pd.read_csv(input_path, engine='python')
    df_processed = preprocess_dataset(df)
    save_processed_data(df_processed, output_path)

## 1. Load Data
بنحمّل `text_splits.csv` اللي صاحبك حفظه من الـ Feature Extraction

In [ ]:
DATA_PATH = '/kaggle/working/clean_reviews.csv'

df = pd.read_csv(DATA_PATH, engine='python')

# rename
df = df.rename(columns={
    'Sentiment': 'label',
    'Cleaned Review': 'text'
})

from sklearn.model_selection import train_test_split

# split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_df['split'] = 'train'
test_df['split'] = 'test'

df = pd.concat([train_df, test_df])

# نخلي بس الأعمدة المطلوبة
df = df[['split', 'text', 'label']]

# reset index (اختياري بس أنضف)
df = df.reset_index(drop=True)

print(df.head())
print(df['split'].value_counts())
print(df['label'].value_counts())

In [ ]:
# ── فصل Train و Test ─────────────────────────────────────────────────────
train_df = df[df['split'] == 'train'].reset_index(drop=True)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)

X_train, y_train = train_df['text'].tolist(), train_df['label'].tolist()
X_test,  y_test  = test_df['text'].tolist(),  test_df['label'].tolist()

print(f'Train : {len(X_train):,} reviews')
print(f'Test  : {len(X_test):,}  reviews')

# ── Class Imbalance ───────────────────────────────────────────────────────
# الداتا فيها imbalance (71% Negative, 29% Positive)
# هنعمل class weights عشان الموديل ميتحيزش للـ Negative
classes = np.array([0, 1])
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights = torch.tensor(weights, dtype=torch.float).to(DEVICE)
print(f'\nClass weights: Negative={weights[0]:.3f} | Positive={weights[1]:.3f}')

## 2. Tokenization
BERT محتاج tokenizer خاص بيه مش بس split على spaces

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN    = 128   # أقصر من 512 عشان أسرع على CPU

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
print(f'Tokenizer loaded: {MODEL_NAME} ✓')

# مثال على الـ tokenization
sample = X_train[0]
encoded = tokenizer(sample, max_length=MAX_LEN, truncation=True, padding='max_length')
print(f'\nSample text    : {sample[:80]}...')
print(f'Input IDs len  : {len(encoded["input_ids"])}')
print(f'Attention mask : {encoded["attention_mask"][:10]}...')

## 3. PyTorch Dataset & DataLoader

In [ ]:
class ReviewDataset(Dataset):
    """
    Custom PyTorch Dataset للـ Amazon Reviews.
    بيحوّل كل review لـ tensors يقدر BERT ياخدها.
    """
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text  = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length      = self.max_len,
            padding         = 'max_length',
            truncation      = True,
            return_tensors  = 'pt'
        )
        return {
            'input_ids'      : encoding['input_ids'].squeeze(0),
            'attention_mask' : encoding['attention_mask'].squeeze(0),
            'label'          : torch.tensor(label, dtype=torch.long)
        }


# ── إنشاء الـ Datasets ────────────────────────────────────────────────────
train_dataset = ReviewDataset(X_train, y_train, tokenizer, MAX_LEN)
test_dataset  = ReviewDataset(X_test,  y_test,  tokenizer, MAX_LEN)

# ── إنشاء الـ DataLoaders ─────────────────────────────────────────────────
# BATCH_SIZE = 16 مناسب لـ CPU, لو عندك GPU ممكن ترفعيه لـ 32
BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches : {len(train_loader):,}')
print(f'Test  batches : {len(test_loader):,}')
print(f'Batch size    : {BATCH_SIZE}')

## 4. Build Model

In [ ]:
# ── تحميل DistilBERT مع Classification Head ───────────────────────────────
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels = 2
)
model = model.to(DEVICE)

# ── عدد البارامترات ───────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Model loaded on: {DEVICE} ✓')

## 5. Training Setup

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────
EPOCHS    = 3       # 3 epochs كافية لـ BERT
LR        = 2e-5    # Learning rate المعياري لـ fine-tuning BERT

# ── Optimizer ────────────────────────────────────────────────────────────
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

# ── Scheduler (Learning Rate Warmup) ─────────────────────────────────────
total_steps   = len(train_loader) * EPOCHS
warmup_steps  = total_steps // 10   # 10% warmup

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps
)

# ── Loss Function مع Class Weights ───────────────────────────────────────
# عشان نتعامل مع الـ imbalance
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

print(f'Epochs         : {EPOCHS}')
print(f'Learning rate  : {LR}')
print(f'Total steps    : {total_steps:,}')
print(f'Warmup steps   : {warmup_steps:,}')
print('Training setup complete ✓')

## 6. Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, loss_fn, device):
    """
    تدريب الموديل على epoch واحدة.
    بترجع average loss و accuracy للـ epoch.
    """
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch_idx, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits

        loss = loss_fn(logits, labels)
        loss.backward()

        # Gradient clipping عشان نمنع exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds       = torch.argmax(logits, dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

        # طباعة كل 100 batch
        if (batch_idx + 1) % 100 == 0:
            print(f'  Batch {batch_idx+1}/{len(loader)} | '
                  f'Loss: {loss.item():.4f}')

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy


def evaluate(model, loader, loss_fn, device):
    """
    تقييم الموديل على test set.
    بترجع loss, accuracy, predictions, true labels.
    """
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits  = outputs.logits

            loss        = loss_fn(logits, labels)
            total_loss += loss.item()

            preds    = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy, np.array(all_preds), np.array(all_labels)


print('Training functions defined ✓')

In [ ]:
# ── Training Loop الرئيسي ────────────────────────────────────────────────
history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
best_acc = 0

print('=' * 60)
print('Starting BERT Fine-tuning...')
print(f'Device: {DEVICE} | Epochs: {EPOCHS} | Batch: {BATCH_SIZE}')
print('=' * 60)

for epoch in range(1, EPOCHS + 1):
    print(f'\nEpoch {epoch}/{EPOCHS}')
    print('-' * 40)

    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, scheduler, loss_fn, DEVICE
    )
    test_loss, test_acc, preds, labels = evaluate(
        model, test_loader, loss_fn, DEVICE
    )

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)

    print(f'  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')
    print(f'  Test  Loss: {test_loss:.4f} | Test  Acc: {test_acc:.4f}')

    # حفظ أحسن موديل
    if test_acc > best_acc:
        best_acc = test_acc
        os.makedirs('./outputs/models', exist_ok=True)
        torch.save(model.state_dict(), './outputs/models/bert_best.pt')
        print(f'  ✓ Best model saved! (acc={best_acc:.4f})')

print('\n' + '=' * 60)
print(f'Training Complete! Best Test Accuracy: {best_acc:.4f}')
print('=' * 60)

In [ ]:
import os

print(os.listdir('/kaggle/working/outputs/models'))

## 7. Evaluation & Metrics

In [ ]:
# ── تحميل أحسن موديل وعمل Final Evaluation ───────────────────────────────
model.load_state_dict(torch.load('/kaggle/working/outputs/models/bert_best.pt', map_location=DEVICE))

_, final_acc, final_preds, final_labels = evaluate(
    model, test_loader, loss_fn, DEVICE
)

print('=' * 60)
print('BERT — Final Evaluation on Test Set')
print('=' * 60)
print(f'Accuracy: {final_acc:.4f} ({final_acc*100:.2f}%)')
print()
print(classification_report(
    final_labels, final_preds,
    target_names=['Negative', 'Positive']
))

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(final_labels, final_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative', 'Positive'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('BERT — Confusion Matrix', fontsize=13, fontweight='bold')

# Training History
epochs_range = range(1, EPOCHS + 1)
axes[1].plot(epochs_range, history['train_acc'], 'o-', label='Train Acc', color='steelblue')
axes[1].plot(epochs_range, history['test_acc'],  's-', label='Test Acc',  color='tomato')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('BERT — Accuracy per Epoch', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs('../src/figures', exist_ok=True)
plt.savefig('../src/figures/bert_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✓')

## 8. Insights Extraction
استخراج insights من نتائج الموديل (المطلوب في المشروع)

In [ ]:
# ── إنشاء Results DataFrame ───────────────────────────────────────────────
results_df = test_df.copy()
results_df['predicted']  = final_preds
results_df['correct']    = (results_df['label'] == results_df['predicted'])
results_df['sentiment_label']   = results_df['label'].map({0: 'Negative', 1: 'Positive'})
results_df['predicted_label']   = results_df['predicted'].map({0: 'Negative', 1: 'Positive'})

print('Results DataFrame created ✓')
results_df.head()

In [ ]:
# ── Statistics ────────────────────────────────────────────────────────────
total     = len(results_df)
pos_count = (results_df['predicted'] == 1).sum()
neg_count = (results_df['predicted'] == 0).sum()

print('=' * 50)
print('BERT Prediction Statistics')
print('=' * 50)
print(f'Total reviews   : {total:,}')
print(f'Positive (pred) : {pos_count:,} ({pos_count/total*100:.1f}%)')
print(f'Negative (pred) : {neg_count:,} ({neg_count/total*100:.1f}%)')

# غلطات الموديل — فين بيغلط أكتر؟
errors = results_df[~results_df['correct']]
fp = len(errors[errors['label'] == 0])  # Negative اتتنبأ بـ Positive
fn = len(errors[errors['label'] == 1])  # Positive اتتنبأ بـ Negative
print(f'\nFalse Positives : {fp} (Negative predicted as Positive)')
print(f'False Negatives : {fn} (Positive predicted as Negative)')

In [ ]:
# ── Sample Predictions مع Reasoning ──────────────────────────────────────
# المطلوب في المشروع: Negative → Reason

print('=' * 60)
print('Sample Predictions with Reasoning')
print('=' * 60)

# keywords لكل sentiment من TF-IDF analysis
positive_keywords = ['great', 'good', 'love', 'excellent', 'best', 'amazing', 'fast', 'easy']
negative_keywords = ['slow', 'bad', 'terrible', 'never', 'refund', 'cancel', 'delay', 'broken',
                     'worst', 'awful', 'disappointed', 'damaged', 'missing', 'lost']

def extract_reason(text, sentiment):
    """استخراج السبب من النص بناءً على الـ keywords"""
    words = text.lower().split()
    if sentiment == 0:  # Negative
        found = [w for w in words if w in negative_keywords]
        return f"Reason: '{', '.join(found[:3])}' mentioned" if found else "Reason: negative tone detected"
    else:  # Positive
        found = [w for w in words if w in positive_keywords]
        return f"Reason: '{', '.join(found[:3])}' mentioned" if found else "Reason: positive tone detected"

# عرض 5 أمثلة
sample_results = results_df.sample(5, random_state=42)
for _, row in sample_results.iterrows():
    sentiment = row['predicted_label']
    reason    = extract_reason(row['text'], row['predicted'])
    correct   = '✓' if row['correct'] else '✗'
    print(f"\n{correct} Review : {str(row['text'])[:80]}...")
    print(f"   → {sentiment} | {reason}")

In [ ]:
# ── Common Patterns ───────────────────────────────────────────────────────
from collections import Counter

def get_top_keywords(texts, keywords_list, top_n=10):
    """أكتر الـ keywords ظهوراً في النصوص"""
    counter = Counter()
    for text in texts:
        words = text.lower().split()
        for word in words:
            if word in keywords_list:
                counter[word] += 1
    return counter.most_common(top_n)

neg_texts = results_df[results_df['predicted'] == 0]['text']
pos_texts = results_df[results_df['predicted'] == 1]['text']

top_neg = get_top_keywords(neg_texts, negative_keywords)
top_pos = get_top_keywords(pos_texts, positive_keywords)

print('Common Patterns — Negative Reviews:')
for word, count in top_neg:
    print(f'  {word:<15} → {count:,} mentions')

print('\nCommon Patterns — Positive Reviews:')
for word, count in top_pos:
    print(f'  {word:<15} → {count:,} mentions')

## 9. Comparison with Baseline
مقارنة BERT مع الـ Baseline Model (Logistic Regression / Naive Bayes)

In [ ]:
# # ⚠️ غيّري أرقام الـ Baseline بالنتايج الحقيقية من الجزء التالت
# # الأرقام دي placeholder

# comparison = pd.DataFrame({
#     'Model'     : ['Logistic Regression (TF-IDF)', 'Naive Bayes (TF-IDF)', 'BERT (DistilBERT)'],
#     'Accuracy'  : [0.000, 0.000, final_acc],      # ← ضيفي أرقام الـ baseline
#     'Precision' : [0.000, 0.000, 0.000],           # ← ضيفي من classification_report
#     'Recall'    : [0.000, 0.000, 0.000],
#     'F1-Score'  : [0.000, 0.000, 0.000],
# })

# print('Model Comparison Table')
# print('(Replace 0.000 with actual baseline results from Task 3)')
# display(comparison)

# # حفظ الـ comparison
# os.makedirs('../outputs/results', exist_ok=True)
# comparison.to_csv('../outputs/results/model_comparison.csv', index=False)
# print('Comparison saved ✓')

## 10. Summary

In [ ]:
print('=' * 60)
print('BERT Model — Final Summary')
print('=' * 60)
print(f'Model          : DistilBERT (distilbert-base-uncased)')
print(f'Max Seq Length : {MAX_LEN}')
print(f'Epochs         : {EPOCHS}')
print(f'Batch Size     : {BATCH_SIZE}')
print(f'Learning Rate  : {LR}')
print(f'Device         : {DEVICE}')
print(f'Test Accuracy  : {final_acc:.4f} ({final_acc*100:.2f}%)')
print('=' * 60)
print('Outputs saved to ../outputs/')
print('  models/bert_best.pt')
print('  results/model_comparison.csv')
print('  figures/bert_evaluation.png (via src/figures)')

# Feature Extraction 

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

import numpy as np
import gensim

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    df['text'],   
    df['label'],
    test_size=0.2,
    random_state=42
)

print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

# TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Word2Vec
# Tokenize sentences for Word2Vec
sentences_train = [text.split() for text in X_train]
sentences_test = [text.split() for text in X_test]

w2v_model = gensim.models.Word2Vec(
    sentences_train,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

def get_sentence_vector(words, model):
    valid_words = [word for word in words if word in model.wv]
    if valid_words:
        return np.mean(model.wv[valid_words], axis=0)
    else:
        return np.zeros(model.vector_size)

X_train_w2v = np.array([
    get_sentence_vector(words, w2v_model)
    for words in sentences_train
])

X_test_w2v = np.array([
    get_sentence_vector(words, w2v_model)
    for words in sentences_test
])

# Model Training and Evaluation

In [ ]:
# Train LR on TF-IDF
lr_tfidf = LogisticRegression(max_iter=1000)
lr_tfidf.fit(X_train_tfidf, y_train)
preds_tfidf = lr_tfidf.predict(X_test_tfidf)
print("TF-IDF Model Accuracy:", accuracy_score(y_test, preds_tfidf))
print(classification_report(y_test, preds_tfidf))

# Train LR on Word2Vec
lr_w2v = LogisticRegression(max_iter=1000)
lr_w2v.fit(X_train_w2v, y_train)
preds_w2v = lr_w2v.predict(X_test_w2v)
print("Word2Vec Model Accuracy:", accuracy_score(y_test, preds_w2v))
print(classification_report(y_test, preds_w2v))

In [ ]:
# ── Evaluation Statistics for Linear Models (TF-IDF & Word2Vec) ─────────────
from sklearn.metrics import accuracy_score, precision_score, classification_report, confusion_matrix

# ── TF-IDF (Logistic Regression — Linear Model) ──────────────────────────────
correct_tfidf   = (preds_tfidf == y_test).sum()
incorrect_tfidf = (preds_tfidf != y_test).sum()
acc_tfidf       = accuracy_score(y_test, preds_tfidf)
prec_tfidf      = precision_score(y_test, preds_tfidf, average='weighted')

print("=" * 55)
print("Linear Model (TF-IDF + Logistic Regression) — Test Stats")
print("=" * 55)
print(f"  Correct Predictions   : {correct_tfidf:,}")
print(f"  Incorrect Predictions : {incorrect_tfidf:,}")
print(f"  Accuracy              : {acc_tfidf:.4f}  ({acc_tfidf*100:.2f}%)")
print(f"  Precision (weighted)  : {prec_tfidf:.4f}")
print()
print(classification_report(y_test, preds_tfidf, target_names=['Negative', 'Positive']))

# ── Word2Vec (Logistic Regression — Linear Model) ────────────────────────────
correct_w2v   = (preds_w2v == y_test).sum()
incorrect_w2v = (preds_w2v != y_test).sum()
acc_w2v       = accuracy_score(y_test, preds_w2v)
prec_w2v      = precision_score(y_test, preds_w2v, average='weighted')

print("=" * 55)
print("Linear Model (Word2Vec + Logistic Regression) — Test Stats")
print("=" * 55)
print(f"  Correct Predictions   : {correct_w2v:,}")
print(f"  Incorrect Predictions : {incorrect_w2v:,}")
print(f"  Accuracy              : {acc_w2v:.4f}  ({acc_w2v*100:.2f}%)")
print(f"  Precision (weighted)  : {prec_w2v:.4f}")
print()
print(classification_report(y_test, preds_w2v, target_names=['Negative', 'Positive']))


# Saving and Loading Models

In [ ]:
import pickle
os.makedirs("models", exist_ok=True)

# Save TF-IDF and Model
with open("models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vectorizer, f)
with open("models/lr_tfidf.pkl", "wb") as f:
    pickle.dump(lr_tfidf, f)

# Save Word2Vec and Model
w2v_model.save("models/w2v_model.gensim")
with open("models/lr_w2v.pkl", "wb") as f:
    pickle.dump(lr_w2v, f)

print("Models saved successfully in the 'models' directory.")

# Gradio Interface

In [ ]:
import gradio as gr
import numpy as np
import torch
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from collections import Counter
import re

warnings.filterwarnings("ignore")

# Ensure BERT model is in evaluation mode
if 'model' in globals():
    model.eval()

# Global session tracker for statistics and common patterns
session = {
    'total': 0,
    'positive': 0,
    'negative': 0,
    'all_tokens': [],
    'correct': 0,
    'incorrect': 0,
    'all_preds': [],
    'all_labels': []
}

# Define a list of sentiment-heavy keywords for "Reason" extraction
INSIGHT_KEYWORDS = {
    'negative': ['slow', 'bad', 'terrible', 'refund', 'broken', 'worst', 'expensive', 'late', 'poor'],
    'positive': ['great', 'fast', 'love', 'excellent', 'amazing', 'best', 'good', 'cheap', 'helpful']
}

def extract_insights(text, label_idx):
    """Extracts the reason and important keywords from the text."""
    text = text.lower()
    sentiment = 'positive' if label_idx == 1 else 'negative'
    found_keywords = [word for word in INSIGHT_KEYWORDS[sentiment] if word in text]
    if found_keywords:
        reason = f"{sentiment.capitalize()} '{found_keywords[0]}' mentioned"
    else:
        reason = f"General {sentiment} tone detected"
    return reason, ", ".join(found_keywords) if found_keywords else "N/A"

def build_confusion_matrix_plot(preds_list, labels_list):
    """Build and return a confusion matrix figure."""
    if len(preds_list) < 2:
        fig, ax = plt.subplots(figsize=(4, 3))
        ax.text(0.5, 0.5, 'Not enough data\nfor confusion matrix',
                ha='center', va='center', fontsize=11)
        ax.axis('off')
        return fig
    cm = confusion_matrix(labels_list, preds_list, labels=[0, 1])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative', 'Positive'])
    fig, ax = plt.subplots(figsize=(4, 3))
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title('Confusion Matrix (Session)', fontsize=11, fontweight='bold')
    plt.tight_layout()
    return fig

def analyze_review(review_text, model_choice, true_label_str):
    if not review_text or not str(review_text).strip():
        fig = build_confusion_matrix_plot([], [])
        return "N/A", "N/A", "N/A", "N/A", "Please enter a review.", "N/A", fig

    try:
        cleaned_review = preprocess_text(str(review_text))

        # 1. Classify Sentiment
        if model_choice == "BERT" and 'model' in globals():
            encoding = tokenizer(cleaned_review, max_length=128, padding='max_length',
                                 truncation=True, return_tensors='pt')
            input_ids = encoding['input_ids'].to(DEVICE)
            attention_mask = encoding['attention_mask'].to(DEVICE)
            with torch.no_grad():
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                probs = torch.softmax(outputs.logits, dim=1)[0].cpu().numpy()
                pred = int(np.argmax(probs))
        elif model_choice == "Linear (TF-IDF)":
            features = tfidf_vectorizer.transform([cleaned_review])
            pred = int(lr_tfidf.predict(features)[0])
            probs = lr_tfidf.predict_proba(features)[0]
        else:  # Linear (W2V)
            features = np.array([get_sentence_vector(cleaned_review.split(), w2v_model)])
            pred = int(lr_w2v.predict(features)[0])
            probs = lr_w2v.predict_proba(features)[0]

        # 2. Extract Reason and Keywords
        reason, keywords = extract_insights(review_text, pred)

        # 3. Update Session Statistics
        session['total'] += 1
        session['all_preds'].append(pred)
        if pred == 1:
            session['positive'] += 1
        else:
            session['negative'] += 1

        # Track words for "Common Patterns"
        words = re.findall(r'\w+', cleaned_review.lower())
        session['all_tokens'].extend(words)

        # Handle true label for correct/incorrect tracking
        true_label_map = {"Positive": 1, "Negative": 0, "Unknown": None}
        true_label = true_label_map.get(true_label_str, None)
        if true_label is not None:
            session['all_labels'].append(true_label)
            if pred == true_label:
                session['correct'] += 1
            else:
                session['incorrect'] += 1
            correct_str = f"Correct: {session['correct']} | Incorrect: {session['incorrect']}"
        else:
            session['all_labels'].append(pred)  # treat pred as label when unknown
            correct_str = "True label not provided — correct/incorrect not tracked"

        # Statistics
        pos_pct = (session['positive'] / session['total']) * 100
        neg_pct = (session['negative'] / session['total']) * 100
        stats_str = (f"Positive: {pos_pct:.1f}% | Negative: {neg_pct:.1f}% "
                     f"(Total: {session['total']})")

        common_words = [w for w, c in Counter(session['all_tokens']).most_common(5)]
        patterns = ", ".join(common_words)

        sentiment_label = "Positive (✓)" if pred == 1 else "Negative (✘)"

        # Build confusion matrix
        cm_fig = build_confusion_matrix_plot(session['all_preds'], session['all_labels'])

        return sentiment_label, reason, keywords, patterns, stats_str, correct_str, cm_fig

    except Exception as e:
        fig = build_confusion_matrix_plot([], [])
        return "Error", str(e), "N/A", "N/A", "N/A", "N/A", fig


# ── Build Gradio UI ───────────────────────────────────────────────────────────
with gr.Blocks(title="Sentiment Insight System", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📊 Sentiment Analysis & Insight Extraction")
    gr.Markdown(
        "Classifies sentiment using **Linear Models** (Logistic Regression) or BERT, "
        "and extracts reasons behind predictions."
    )

    with gr.Row():
        with gr.Column():
            review_input = gr.Textbox(
                label="Review Text", lines=4,
                placeholder="e.g., The service was slow and delivery took forever."
            )
            model_radio = gr.Radio(
                ["Linear (TF-IDF)", "Linear (W2V)", "BERT"],
                value="Linear (TF-IDF)",
                label="Model Selection  (Linear = Logistic Regression)"
            )
            true_label_input = gr.Radio(
                ["Positive", "Negative", "Unknown"],
                value="Unknown",
                label="True Label (for accuracy tracking)"
            )
            analyze_btn = gr.Button("Extract Insights", variant="primary")

        with gr.Column():
            out_sentiment = gr.Textbox(label="Sentiment Classification")
            out_reason    = gr.Textbox(label="Reason")
            out_keywords  = gr.Textbox(label="Important Keywords")
            out_correct   = gr.Textbox(label="Prediction Stats (Correct / Incorrect)")

    gr.Markdown("---")
    with gr.Row():
        out_patterns = gr.Textbox(label="Common Patterns (Top Words Across Session)")
        out_stats    = gr.Textbox(label="Simple Statistics (% Positive vs Negative)")

    gr.Markdown("### Confusion Matrix (Session)")
    out_cm = gr.Plot(label="Confusion Matrix")

    analyze_btn.click(
        fn=analyze_review,
        inputs=[review_input, model_radio, true_label_input],
        outputs=[out_sentiment, out_reason, out_keywords,
                 out_patterns, out_stats, out_correct, out_cm]
    )

demo.launch(share=True)